# Assignment 9: VAE for Compression and Denoising

**Parts covered:** A (compression ratio), B (reconstruction + per-image MSE), C (denoising), D (latent dim 2 vs 16 vs 32), plus the results table and comparison images.

**Assumption:** Assignment 8's VAE isn't available to me, so this is a standard MLP VAE on MNIST (784 → 512 → 256 → latent). If your A8 model is different, replace the `VAE` class and keep `LATENT_DIMS[0] = 2` so the comparison starts from your original model. Everything else is model-agnostic as long as the model exposes `encode(x) -> (mu, logvar)` and `decode(z)`.

**Outputs saved:** `reconstruction_comparison.png`, `denoising_comparison.png`, `latent_space_2d.png`, `results_table.csv`

In [ ]:
import random, numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import torchvision, torchvision.transforms as T
import matplotlib.pyplot as plt

SEED = 42
def set_seed(s=SEED):
    random.seed(s); np.random.seed(s); torch.manual_seed(s); torch.cuda.manual_seed_all(s)
set_seed()
device = torch.device("cuda" if torch.cuda.is_available() else "mps" if torch.backends.mps.is_available() else "cpu")
print("device:", device)

## 1. Configuration

In [ ]:
LATENT_DIMS = [2, 16, 32]   # assignment: 2 -> larger (16, 32)
EPOCHS      = 20
BATCH       = 128
LR          = 1e-3
NOISE_STD   = 0.3           # Gaussian noise std in [0,1] pixel space (Part C)
NOISE_SWEEP = [0.1, 0.3, 0.5]  # extra: how denoising degrades with noise level

## 2. Data

In [ ]:
tf = T.ToTensor()  # pixels in [0,1]; Bernoulli/BCE decoder expects this range
train_ds = torchvision.datasets.MNIST("./data", train=True,  download=True, transform=tf)
test_ds  = torchvision.datasets.MNIST("./data", train=False, download=True, transform=tf)
train_loader = torch.utils.data.DataLoader(train_ds, batch_size=BATCH, shuffle=True, drop_last=True)
test_loader  = torch.utils.data.DataLoader(test_ds, batch_size=1000)

# 10 test images: first occurrence of each digit 0-9 (deterministic, covers all classes)
idx = [next(i for i, (_, y) in enumerate(test_ds) if y == d) for d in range(10)]
sel_x = torch.stack([test_ds[i][0] for i in idx]).view(10, -1).to(device)
sel_y = [test_ds[i][1] for i in idx]
print("selected test indices:", idx, "labels:", sel_y)

## 3. Model and training

In [ ]:
class VAE(nn.Module):
    def __init__(self, latent):
        super().__init__()
        self.fc1 = nn.Linear(784, 512); self.fc2 = nn.Linear(512, 256)
        self.mu = nn.Linear(256, latent); self.lv = nn.Linear(256, latent)
        self.d1 = nn.Linear(latent, 256); self.d2 = nn.Linear(256, 512); self.d3 = nn.Linear(512, 784)
    def encode(self, x):
        h = F.relu(self.fc2(F.relu(self.fc1(x))))
        return self.mu(h), self.lv(h)
    def decode(self, z):
        return torch.sigmoid(self.d3(F.relu(self.d2(F.relu(self.d1(z))))))
    def forward(self, x):
        mu, lv = self.encode(x)
        z = mu + torch.randn_like(mu) * torch.exp(0.5 * lv)
        return self.decode(z), mu, lv

def vae_loss(recon, x, mu, lv):
    bce = F.binary_cross_entropy(recon, x, reduction="sum")
    kld = -0.5 * torch.sum(1 + lv - mu.pow(2) - lv.exp())
    return (bce + kld) / x.size(0)

def train_vae(latent, epochs=EPOCHS):
    set_seed()
    m = VAE(latent).to(device); opt = torch.optim.Adam(m.parameters(), lr=LR)
    for ep in range(1, epochs + 1):
        tot = 0.0
        for x, _ in train_loader:
            x = x.view(x.size(0), -1).to(device)
            opt.zero_grad(); r, mu, lv = m(x); l = vae_loss(r, x, mu, lv); l.backward(); opt.step(); tot += l.item()
        if ep == 1 or ep % 5 == 0 or ep == epochs:
            print(f"[latent={latent:2d}] epoch {ep:2d}/{epochs}  loss/sample {tot/len(train_loader):.2f}")
    return m.eval()

models = {d: train_vae(d) for d in LATENT_DIMS}

## 4. Part A: Compression ratio

Counted in **values**, as the assignment specifies: 784 / latent_dim. Because the encoder produces both `mu` and `logvar`, only `mu` is stored/transmitted for compression (the decoder is run on `mu`, the deterministic code).

Bytes are shown too, because the values-based ratio flatters the VAE: raw MNIST pixels are 1 byte each (uint8), while latents are float32 (4 bytes each). Unless you quantise the latent, the true byte ratio is 4x lower than the headline number.

In [ ]:
def compression(latent):
    return dict(
        values_ratio = 784 / latent,
        bytes_ratio  = (784 * 1) / (latent * 4),   # uint8 pixels vs float32 latent
    )
for d in LATENT_DIMS:
    c = compression(d)
    print(f"latent {d:2d}: 784 -> {d:2d} values | ratio {c['values_ratio']:6.1f}:1 (values) | {c['bytes_ratio']:5.1f}:1 (bytes, float32 latent)")

## 5. Part B: Reconstruction (10 test images, per-image MSE)

In [ ]:
@torch.no_grad()
def reconstruct(model, x):
    mu, _ = model.encode(x)          # deterministic: use the mean, not a sampled z
    return model.decode(mu)

def mse_per_image(a, b): return ((a - b) ** 2).mean(dim=1)
def psnr(mse):           return 10 * torch.log10(1.0 / mse.clamp_min(1e-10))   # pixel range [0,1]

recons = {d: reconstruct(models[d], sel_x) for d in LATENT_DIMS}

rows = []
for d in LATENT_DIMS:
    m = mse_per_image(recons[d], sel_x)
    for i in range(10):
        rows.append(dict(latent=d, image=i, digit=sel_y[i], mse=m[i].item(), psnr=psnr(m[i]).item()))
df_img = pd.DataFrame(rows)
print(df_img.pivot(index="digit", columns="latent", values="mse").round(5))
print("\nmean MSE over the 10 images:"); print(df_img.groupby("latent").mse.mean().round(5))

# comparison image: originals on top, reconstruction per latent dim below
fig, axs = plt.subplots(1 + len(LATENT_DIMS), 10, figsize=(15, 1.6 * (1 + len(LATENT_DIMS))))
for j in range(10):
    axs[0, j].imshow(sel_x[j].view(28, 28).cpu(), cmap="gray"); axs[0, j].axis("off")
    for r, d in enumerate(LATENT_DIMS, start=1):
        axs[r, j].imshow(recons[d][j].view(28, 28).cpu(), cmap="gray"); axs[r, j].axis("off")
        axs[r, j].set_title(f"{mse_per_image(recons[d][j:j+1], sel_x[j:j+1]).item():.4f}", fontsize=7)
axs[0, 0].set_title("Original", fontsize=8, loc="left")
for r, d in enumerate(LATENT_DIMS, start=1): axs[r, 0].set_title(f"latent={d}  (title = MSE)", fontsize=8, loc="left")
plt.tight_layout(); plt.savefig("reconstruction_comparison.png", dpi=150); plt.show()

## 6. Part C: Denoising

**Read this before interpreting results.** The VAE was trained on *clean* data only, not as a denoising autoencoder. Any denoising is implicit: the encoder maps the noisy image to a point near the learned digit manifold, and the decoder emits a clean-looking digit from it. This works only while the noise is small enough that the encoder still lands near the right region. Expect it to fail at high noise, and expect the output to be a *plausible digit*, not necessarily the *correct* one.

Noise is added in [0,1] pixel space and clipped back to [0,1]. Two MSE numbers are reported against the clean image: the noisy input (the baseline) and the VAE output. Denoising only counts as working if VAE-output MSE < noisy-input MSE.

In [ ]:
def add_noise(x, std, seed=SEED):
    g = torch.Generator(device=x.device).manual_seed(seed)
    return (x + std * torch.randn(x.shape, generator=g, device=x.device)).clamp(0, 1)

noisy = add_noise(sel_x, NOISE_STD)
denoised = {d: reconstruct(models[d], noisy) for d in LATENT_DIMS}

print(f"noise std = {NOISE_STD}")
print(f"noisy input vs clean      MSE: {mse_per_image(noisy, sel_x).mean().item():.5f}")
for d in LATENT_DIMS:
    print(f"denoised (latent={d:2d}) vs clean MSE: {mse_per_image(denoised[d], sel_x).mean().item():.5f}")

fig, axs = plt.subplots(2 + len(LATENT_DIMS), 10, figsize=(15, 1.6 * (2 + len(LATENT_DIMS))))
for j in range(10):
    axs[0, j].imshow(sel_x[j].view(28, 28).cpu(), cmap="gray"); axs[0, j].axis("off")
    axs[1, j].imshow(noisy[j].view(28, 28).cpu(), cmap="gray"); axs[1, j].axis("off")
    for r, d in enumerate(LATENT_DIMS, start=2):
        axs[r, j].imshow(denoised[d][j].view(28, 28).cpu(), cmap="gray"); axs[r, j].axis("off")
        axs[r, j].set_title(f"{mse_per_image(denoised[d][j:j+1], sel_x[j:j+1]).item():.4f}", fontsize=7)
axs[0, 0].set_title("Clean", fontsize=8, loc="left"); axs[1, 0].set_title(f"Noisy (std={NOISE_STD})", fontsize=8, loc="left")
for r, d in enumerate(LATENT_DIMS, start=2): axs[r, 0].set_title(f"Denoised, latent={d}  (title = MSE vs clean)", fontsize=8, loc="left")
plt.tight_layout(); plt.savefig("denoising_comparison.png", dpi=150); plt.show()

### Extra: denoising vs noise level (full 10,000-image test set, not just 10 images)

In [ ]:
test_x = torch.cat([x.view(x.size(0), -1) for x, _ in test_loader]).to(device)

sweep = []
for std in NOISE_SWEEP:
    nz = add_noise(test_x, std)
    row = {"noise_std": std, "noisy_input_MSE": mse_per_image(nz, test_x).mean().item()}
    for d in LATENT_DIMS:
        row[f"latent_{d}_MSE"] = mse_per_image(reconstruct(models[d], nz), test_x).mean().item()
    sweep.append(row)
df_sweep = pd.DataFrame(sweep).round(5); print(df_sweep.to_string(index=False))

## 7. Part D: Latent dimension comparison and results table

In [ ]:
# Full test-set metrics (10 images is too small a sample to conclude anything; these are the numbers to trust)
full = {}
for d in LATENT_DIMS:
    rec = reconstruct(models[d], test_x)
    nz = add_noise(test_x, NOISE_STD)
    den = reconstruct(models[d], nz)
    full[d] = dict(
        mse_full=mse_per_image(rec, test_x).mean().item(),
        psnr_full=psnr(mse_per_image(rec, test_x)).mean().item(),
        denoise_mse=mse_per_image(den, test_x).mean().item(),
    )
noisy_mse = mse_per_image(add_noise(test_x, NOISE_STD), test_x).mean().item()

table = pd.DataFrame([{
    "Latent dim": d,
    "Compression ratio (values)": f"{784/d:.1f}:1",
    "Compression ratio (bytes, float32)": f"{(784)/(d*4):.1f}:1",
    "Recon MSE (10 imgs)": round(df_img[df_img.latent == d].mse.mean(), 5),
    "Recon MSE (10k test)": round(full[d]["mse_full"], 5),
    "PSNR dB (10k test)": round(full[d]["psnr_full"], 2),
    f"Denoise MSE @std={NOISE_STD} (noisy input = {noisy_mse:.4f})": round(full[d]["denoise_mse"], 5),
} for d in LATENT_DIMS])
table["Reconstruction quality"] = ["<fill in after viewing reconstruction_comparison.png>"] * len(LATENT_DIMS)
table["Denoising observations"] = ["<fill in after viewing denoising_comparison.png>"] * len(LATENT_DIMS)
table.to_csv("results_table.csv", index=False)
pd.set_option("display.max_colwidth", None); pd.set_option("display.width", 250)
display(table) if "display" in globals() else print(table.to_string(index=False))

### 2-D latent space (only meaningful for latent=2)

In [ ]:
with torch.no_grad():
    mu2, _ = models[LATENT_DIMS[0]].encode(test_x[:5000])
ys = test_ds.targets[:5000].numpy()
plt.figure(figsize=(6, 5)); sc = plt.scatter(mu2[:, 0].cpu(), mu2[:, 1].cpu(), c=ys, cmap="tab10", s=3)
plt.colorbar(sc, ticks=range(10)); plt.title("Latent space (dim=2), test set"); plt.xlabel("z1"); plt.ylabel("z2")
plt.savefig("latent_space_2d.png", dpi=150, bbox_inches="tight"); plt.show()

## 8. Written observations (fill in from YOUR results)

**Compression:** *[State 784/latent per dimension. Then state the byte-level ratio and that the VAE is lossy: the MSE column is the price paid.]*

**Reconstruction:** *[Does MSE fall as latent goes 2 → 16 → 32? Expect a large drop from 2 to 16 and a small one from 16 to 32. If 32 isn't clearly better than 16, say so: diminishing returns, and the extra latent dimensions may be unused (posterior collapse on some dimensions). Describe how the 2-D reconstructions look (typically blurry, digits confusable) versus 16/32.]*

**Denoising:** *[Compare denoised MSE to noisy-input MSE from the sweep table. State at which noise level, if any, the VAE stops helping. Note the VAE was not trained for denoising, so describe whether failures look like blur or like a wrong digit. Does larger latent help or hurt denoising? Higher capacity can reproduce noise more faithfully, so a larger latent is not guaranteed to denoise better. Report what your numbers show.]*

**Caveats to state:** 10 images is a tiny sample (the 10k-image columns are the reliable numbers); MSE correlates imperfectly with perceptual quality; single seed, no variance estimate.